In [1]:
%cd ../
%load_ext autoreload
%autoreload 2

/Users/matthaei/Documents/code/python/bachelor-project


In [2]:
MIGRATE_DATABASE = True

In [3]:
from src.measurements.measurement_service import MeasurementService
from src.weather_stations.weather_station_service import WeatherStationService
from src.model.variant.bilstm_model import BiLSTMModel
from src.prediction.prediction_service import PredictionService
from src.database.database_service import DatabaseService
from src.model.model_service import ModelService
from omegaconf import DictConfig, OmegaConf
from hydra import compose, initialize_config_dir
import os
from datetime import datetime

/Users/matthaei/Documents/code/python/bachelor-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Initialize Hydra configuration
config_dir = os.path.abspath("./conf")

# Initialize Hydra with the config directory
with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(config_name="config")

In [5]:
database_service = DatabaseService(cfg)

if MIGRATE_DATABASE:
    database_service.create_tables()

2025-09-13 12:52:37.343 | INFO     | src.database.database_service:create_tables:22 - Tables created


In [6]:
weather_station_service = WeatherStationService(cfg, database_service)
weather_stations_df = weather_station_service.load_from_database()

2025-09-13 12:52:40.682 | INFO     | src.weather_stations.weather_station_data_provider:load_from_database:228 - Loaded 50 weather stations from database


In [7]:
measurement_service = MeasurementService(cfg, database_service, weather_stations_df)

In [8]:
model_service = ModelService(cfg, database_service, measurement_service)

In [9]:
lstm = BiLSTMModel()

In [10]:
lstm.load('models/good_lstm/')

In [11]:
model_service.attach_model(lstm)

## Start

In [13]:
prediction_service = PredictionService(cfg, database_service, measurement_service, model_service)

In [16]:
df = prediction_service.predict_measurements()

2025-09-13 12:54:11.983 | INFO     | src.measurements.measurement_service:load_all_recent_measurements_from_database:121 - Loading all recent measurements from database
2025-09-13 12:54:13.128 | INFO     | src.measurements.measurement_data_provider:load_measurements_from_database:240 - Loaded chunk of 3456 rows (total so far: 3456)
2025-09-13 12:54:13.219 | INFO     | src.measurements.measurement_data_provider:load_measurements_from_database:249 - Loaded 3456 measurements from database
2025-09-13 12:54:13.220 | INFO     | src.measurements.measurement_service:load_all_recent_measurements_from_database:123 - Loaded 3456 recent measurements from database
2025-09-13 12:54:13.803 | INFO     | src.prediction.prediction_data_provider:save_predictions_to_database:19 - Upserted 200 predictions to database
2025-09-13 12:54:13.926 | INFO     | src.prediction.prediction_data_provider:save_predictions_to_database:19 - Upserted 200 predictions to database
2025-09-13 12:54:14.062 | INFO     | src.pre